In [1]:
# %% [markdown]
# # 03b — Sparse KAN Experiment (3-Seed Reproducibility)
#
# THE CORE THESIS CONTRIBUTION.
#
# Domain-structured KAN using MaskedKANLinear:
# features → subthemes → themes → output.
# Structural sparsity from taxonomy. Active edge and parameter counts are
# read directly from the model at runtime — no hardcoded numbers.
#
# ═══════════════════════════════════════════════════════════════════════════
# MAJOR REVISION — grid adaptation removed, BatchNorm added
# ═══════════════════════════════════════════════════════════════════════════
#
# An earlier version of this notebook called efficient-kan's update_grid()
# at epochs [1, 5, 20] to reposition each layer's B-spline knots to fit
# the data, with a snapshot-and-revert guard protecting against a
# confirmed numerical instability (update_grid()'s internal curve2coeff()
# least-squares refit could produce NaN/inf on legitimate ACTIVE edges,
# independent of grid_eps). Across a real sweep the guard fired on a
# majority of scheduled attempts -- the mechanism was rarely providing any
# benefit while adding real risk and substantial code complexity.
#
# A follow-up measurement of per-node activation ranges (across many
# trained Dense KAN checkpoints, same underlying architecture pattern)
# found the ROOT CAUSE: hidden-layer activation scale varies by orders of
# magnitude depending on dataset, target, and training progress -- from a
# typical spread of 1-5 units to, in the worst observed case, a spread
# exceeding 400. No fixed OR adaptively-repositioned grid can be correct
# for both regimes simultaneously.
#
# THE FIX (now implemented in sparse_kan.py itself, not this notebook):
# nn.BatchNorm1d is inserted between every KAN layer (self.bn1 after
# layer0, self.bn2 after layer1). BatchNorm forces each unit's activation
# toward zero-mean/unit-variance BEFORE the affine step, regardless of
# fan-in or target scale -- the invariant a fixed grid range was always
# silently assuming and never actually had. With that invariant genuinely
# enforced, grid_range=[-5.5, 5.5] is correct for every layer, PERMANENTLY,
# and update_grid() is no longer called anywhere. grid_eps has been
# removed entirely from sparse_kan.py (it was only ever read inside
# update_grid()).
#
# BatchNorm (not LayerNorm) was chosen deliberately: LayerNorm's
# statistics are computed ACROSS UNITS within one sample, entangling every
# subtheme/theme's contribution with every other's -- fatal for this
# architecture's core interpretability claim (isolable per-edge
# contributions). BatchNorm's statistics are computed ACROSS THE BATCH,
# per unit, so unit i's output depends only on unit i's own values. At
# inference it reduces to a fixed per-unit affine map, foldable into the
# adjacent spline for clean symbolic extraction.
#
# ACCORDINGLY, THIS NOTEBOOK NO LONGER CONTAINS:
#   - grid_eps (removed from every SparseKAN.from_taxonomy call)
#   - make_grid_update_fn / _model_has_nonfinite_params / GLOBAL_REVERT_STATS
#   - grid_update_epochs / grid_update_sample / grid_reverts tracking
#   - Sanity Checks 1, 3, 4 (grid_eps forwarding, revert-mechanism unit
#     test, end-to-end grid_update_fn test) -- all made obsolete by the
#     mechanism they were testing no longer existing.
#
# Sanity Check 2 (verify_masking() NaN blind spot) is KEPT -- masking
# itself is unchanged by this revision, and that fix (in sparse_kan.py)
# still needs re-verifying against whatever file is actually on Drive.
#
# ACTIVATION MONITORING (new): training.py gained one small, purely
# additive parameter, epoch_callback (see training.py docstring -- zero
# effect if unset, fires only when verbose=True, i.e. only during the
# final retrain, never during the 70-trial Optuna search). This notebook
# uses it to print each layer's pre/post-BatchNorm max|x| and std on a
# fixed probe batch, live, at the existing log_every cadence -- so a
# config where BatchNorm is not fully containing activation scale is
# visible within minutes, not after the whole sweep finishes.
#
# GRID DESIGN:
#   All three layers: FIXED range [-5.5, 5.5], for the ENTIRE model
#   lifetime. No layer ever calls update_grid(). Layer 0's fixed range was
#   always justified this way (inputs z-scored and clipped at ±5
#   upstream); layers 1/2 are now equally justified, because BatchNorm
#   makes their activation scale controlled the same way layer 0's always
#   was, rather than left to vary with fan-in and target scale.
#
# 3 seeds × 4 splits × 2 datasets × 2 targets = 48 runs.
# Each seed runs the FULL pipeline independently:
#   - Optuna Phase 1: N_TRIALS_NO_L1 trials, no L1
#   - Optuna Phase 2: N_TRIALS trials, with L1
#   - Best across both phases used for final retrain
#   - Evaluation on train/val/test
#
# Target notes:
#   binary:     y_binary (minret_5d_pct < -2.0), BCEWithLogitsLoss,
#               early stop on AUC (uniform across every binary-target
#               model in this project — see training.py docstring).
#   continuous: minret_5d_pct (raw percentage, NOT a z-score),
#               HuberLoss(delta=<per-split value from huber_delta.json>),
#               early stop on R2.
#   Both use Optuna direction="maximize".
#
# COLLAPSE HANDLING: detect-and-flag only, no automatic retry. Not
# checked inline — deferred to the standalone Collapse_Audit notebook,
# run once after every model finishes.
#
# Results save to Drive per-seed as they go — safe against disconnection.
#
# Estimated runtime on T4: likely somewhat faster than the grid-update
# version, since there is no per-scheduled-epoch snapshot/revert overhead
# anymore. See timing cell.
# Runtime disconnects automatically when finished.

# %%
# ── COLAB SETUP ──
!pip install -q git+https://github.com/Blealtan/efficient-kan.git optuna

from google.colab import drive
drive.mount("/content/drive")

import sys
sys.path.insert(0, "/content/drive/MyDrive/Thesis/Code")

# %%
import json
import numpy as np
import pandas as pd
import random
import time
import torch
import optuna
from pathlib import Path

from data_utils import load_split, get_dataloaders, get_device, load_theme_assignment
from training import train_model, save_checkpoint
from evaluation import (
    evaluate_model, save_predictions, compute_calibration,
    load_predictions, run_full_backtest,
)
from sparse_kan import SparseKAN, sparse_kan_edge_l1

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


# ═══════════════════════════════════════════════════════════════════════════
# ACTIVATION MONITORING — live printout during the final retrain only
# ═══════════════════════════════════════════════════════════════════════════
# Uses training.py's epoch_callback hook. Fires only when verbose=True,
# which is the case for the final retrain but NOT the (verbose=False)
# Optuna search phases -- so this is passed to every train_model() call
# below unconditionally, and it will only actually print during the
# final retrain of each config, at the log_every cadence.

def make_activation_callback(probe_batch, device):
    @torch.no_grad()
    def callback(model, epoch):
        model.eval()
        x0 = probe_batch.to(device)

        x1 = model.layer0(x0)
        x1n_preclamp = model.bn1(x1)
        x1n = torch.clamp(x1n_preclamp, min=-5.0, max=5.0)

        x2 = model.layer1(x1n)
        x2n_preclamp = model.bn2(x2)
        x2n = torch.clamp(x2n_preclamp, min=-5.0, max=5.0)

        model.train()

        clamp1_rate = (x1n_preclamp.abs() > 5.0).float().mean().item()
        clamp2_rate = (x2n_preclamp.abs() > 5.0).float().mean().item()

        print(f"    [activations @ epoch {epoch}]  "
              f"L1 pre-BN max|x|={x1.abs().max().item():7.2f} std={x1.std().item():6.3f}  "
              f"post-BN(pre-clamp) max|x|={x1n_preclamp.abs().max().item():5.2f} "
              f"std={x1n_preclamp.std().item():5.3f}  "
              f"clamp_rate={clamp1_rate:.3%}  |  "
              f"L2 pre-BN max|x|={x2.abs().max().item():7.2f} std={x2.std().item():6.3f}  "
              f"post-BN(pre-clamp) max|x|={x2n_preclamp.abs().max().item():5.2f} "
              f"std={x2n_preclamp.std().item():5.3f}  "
              f"clamp_rate={clamp2_rate:.3%}")
    return callback


# ═══════════════════════════════════════════════════════════════════════════
# SANITY CHECK — verify_masking() NaN blind-spot fix, re-verified against
# whatever sparse_kan.py is actually on Drive right now. The grid_eps and
# grid-update-mechanism checks that used to sit here have been removed
# along with the mechanism they tested -- masking is the only thing
# carried over unchanged from the earlier architecture, so it is the only
# thing still worth re-testing here.
# ═══════════════════════════════════════════════════════════════════════════

print("=" * 90)
print("SANITY CHECK: verify_masking() NaN blind spot -- confirming the FIX")
print("=" * 90)
print("  Injecting a real NaN at a masked position and calling the live")
print("  verify_masking(). Before the fix this returned True (silently")
print("  'clear'). After the fix it must return False.")

_fake_tax = pd.DataFrame({
    "column":        ["f1", "f2", "f3", "f4"],
    "subtheme_id":   ["01_01", "01_01", "02_01", "02_01"],
    "subtheme_name": ["SubA", "SubA", "SubB", "SubB"],
    "theme_id":      ["01", "01", "02", "02"],
    "theme_name":    ["ThemeX", "ThemeX", "ThemeY", "ThemeY"],
})
_fcols = ["f1", "f2", "f3", "f4"]

_mb = SparseKAN.from_taxonomy(_fake_tax, _fcols, grid_size=5, spline_order=3,
                              grid_range=[-1, 1])

_inv_positions = (_mb.layer0.mask == 0).nonzero(as_tuple=False)
assert len(_inv_positions) > 0, "Test taxonomy has no masked positions."
_q, _p = _inv_positions[0].tolist()

with torch.no_grad():
    _mb.layer0.base_weight.data[_q, _p] = float("nan")

_verify_result = _mb.verify_masking()
print(f"\n  verify_masking() on a model with an injected masked NaN: {_verify_result}")

assert _verify_result is False, (
    "verify_masking() returned True despite an injected NaN at a masked "
    "position -- the masking NaN-detection fix has NOT been applied to "
    "the sparse_kan.py currently on Drive. STOP. Re-upload the patched "
    "sparse_kan.py before proceeding -- do not run real training against "
    "this file."
)
print("  ✓ verify_masking() correctly detects the injected NaN")

# Also confirm a batch forward pass works end-to-end (exercises
# BatchNorm's requirement of batch size > 1 in train mode, and confirms
# construction no longer accepts/needs grid_eps).
_mb.train()
_x_batch = torch.randn(16, len(_fcols))
_out = _mb(_x_batch)
assert _out.shape == (16, 1)
print("  ✓ Forward pass with BatchNorm in train mode works (batch size 16)")

del _fake_tax, _fcols, _mb, _inv_positions, _q, _p, _verify_result, _x_batch, _out

print("\n" + "=" * 90)
print("SANITY CHECK PASSED")
print("=" * 90)


# ═══════════════════════════════════════════════════════════════════════════
# CONFIGURATION
# ═══════════════════════════════════════════════════════════════════════════

SPLITS_DIR  = Path("/content/drive/MyDrive/Thesis/Data/Stage_5_Model_Ready/04_splits")
THEMES_DIR  = Path("/content/drive/MyDrive/Thesis/Data/Stage_5_Model_Ready/05_themes")
HUBER_DELTA_PATH = Path("/content/drive/MyDrive/Thesis/Data/Stage_5_Model_Ready/03_targets/huber_delta.json")

RESULTS_DIR = Path("/content/drive/MyDrive/Thesis/Data/Results/Dense_vs_Sparse_KAN_v2/sparse_kan")

DATASETS     = ["agg_full_moments", "agg_means"]
TARGET_TYPES = ["binary", "continuous"]
ALL_SPLITS   = ["Split_A", "Split_B", "Split_C", "Split_D"]

SEEDS = [42, 123, 456]

# ── Fixed architecture (not tuned) ──
GRID_SIZE    = 14
SPLINE_ORDER = 3
GRID_RANGE   = [-5.5, 5.5]     # fixed for the model's ENTIRE lifetime,
                               # every layer, no adaptation, ever.

ACTIVATION_PROBE_N = 2048      # rows drawn once per run for the live
                               # activation-monitoring printout above

N_TRIALS       = 40   # with L1
N_TRIALS_NO_L1 = 30   # without L1

with open(HUBER_DELTA_PATH) as f:
    HUBER_DELTAS = json.load(f)["deltas"]

def get_huber_delta(split_name):
    return HUBER_DELTAS[f"{split_name}/market"]


# ═══════════════════════════════════════════════════════════════════════════
# REPRODUCIBILITY
# ═══════════════════════════════════════════════════════════════════════════

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


# ═══════════════════════════════════════════════════════════════════════════
# LOAD TAXONOMIES AND QUICK SANITY CHECK
# ═══════════════════════════════════════════════════════════════════════════

# %%
print("Loading taxonomies...")
taxonomy_dfs = {}
for ds in DATASETS:
    df = load_theme_assignment(ds, THEMES_DIR)
    taxonomy_dfs[ds] = df
    print(f"  {ds}: {len(df)} features, "
          f"{df['subtheme_id'].nunique()} subthemes, "
          f"{df['theme_id'].nunique()} themes")

device = get_device()
architecture_info = {}

for ds in DATASETS:
    data        = load_split("Split_A", ds, SPLITS_DIR)
    model_check = SparseKAN.from_taxonomy(
        taxonomy_dfs[ds], data["feature_cols"],
        grid_size=GRID_SIZE, spline_order=SPLINE_ORDER,
        grid_range=GRID_RANGE,
    ).to(device)

    model_check.train()   # BatchNorm needs train mode + batch>1 to run cleanly here
    x   = torch.randn(4, data["n_features"]).to(device)
    out = model_check(x)
    assert out.shape == (4, 1)

    mask_ok = model_check.verify_masking()
    assert mask_ok, f"{ds}: masking verification failed immediately after construction"

    active_edges  = model_check.count_active_edges()
    total_edges   = model_check.count_total_edges()
    active_params = model_check.count_active_parameters()

    architecture_info[ds] = {
        "n_features":        data["n_features"],
        "n_subthemes":       model_check.n_subthemes,
        "n_themes":          model_check.n_themes,
        "active_edges":      active_edges,
        "total_edges":       total_edges,
        "active_parameters": active_params,
    }

    print(f"  {ds}: {active_edges:,} active edges / {total_edges:,} total  "
          f"({active_params:,} active params) "
          f"— ✓ GPU forward pass OK, masking verified pre-training")
    del model_check, x, out

# ── GPU timing estimate ──
data       = load_split("Split_A", "agg_full_moments", SPLITS_DIR)
model_time = SparseKAN.from_taxonomy(
    taxonomy_dfs["agg_full_moments"], data["feature_cols"],
    grid_size=GRID_SIZE, spline_order=SPLINE_ORDER, grid_range=GRID_RANGE,
).to(device)
model_time.train()
x_bench = torch.randn(256, data["n_features"]).to(device)

if device.type == "cuda":
    torch.cuda.synchronize()
t0 = time.time()
for _ in range(100):
    model_time.zero_grad()
    out  = model_time(x_bench)
    loss = out.sum()
    loss.backward()
if device.type == "cuda":
    torch.cuda.synchronize()
fb_ms = (time.time() - t0) / 100 * 1000

n_batches      = max(1, 3000 // 256)
trial_s        = fb_ms / 1000 * n_batches * 50
total_trials   = (N_TRIALS + N_TRIALS_NO_L1) * len(DATASETS) * len(TARGET_TYPES) * len(ALL_SPLITS)
est_seed_hours = trial_s * total_trials / 3600

print(f"\n  GPU forward+backward (batch=256): {fb_ms:.1f}ms")
print(f"  Estimated per trial: {trial_s:.1f}s")
print(f"  Estimated per seed ({total_trials} trials): {est_seed_hours:.1f} hours")
print(f"  Estimated total (3 seeds): {est_seed_hours * 3:.1f} hours\n")

del model_time, x_bench, data
if device.type == "cuda":
    torch.cuda.empty_cache()


# ═══════════════════════════════════════════════════════════════════════════
# MODEL FACTORIES FOR OPTUNA
# ═══════════════════════════════════════════════════════════════════════════

def make_model_factory_no_l1(feature_cols, taxonomy_df, huber_delta, target_type):
    def factory(trial):
        lr           = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
        weight_decay = trial.suggest_float("weight_decay", 1e-6, 1e-2, log=True)
        batch_size   = trial.suggest_categorical("batch_size", [64, 128, 256])

        model = SparseKAN.from_taxonomy(
            taxonomy_df, feature_cols,
            grid_size=GRID_SIZE, spline_order=SPLINE_ORDER, grid_range=GRID_RANGE,
        )

        train_kwargs = {
            "lr":           lr,
            "weight_decay": weight_decay,
            "n_epochs":     300,
            "patience":     20,
        }
        if target_type == "continuous":
            train_kwargs["huber_delta"] = huber_delta

        return model, train_kwargs
    return factory


def make_model_factory_with_l1(feature_cols, taxonomy_df, huber_delta, target_type):
    def factory(trial):
        lr           = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
        weight_decay = trial.suggest_float("weight_decay", 1e-6, 1e-2, log=True)
        batch_size   = trial.suggest_categorical("batch_size", [64, 128, 256])
        reg_weight   = trial.suggest_float("reg_weight", 1e-7, 1e-3, log=True)

        model = SparseKAN.from_taxonomy(
            taxonomy_df, feature_cols,
            grid_size=GRID_SIZE, spline_order=SPLINE_ORDER, grid_range=GRID_RANGE,
        )

        train_kwargs = {
            "lr":           lr,
            "weight_decay": weight_decay,
            "reg_fn":       sparse_kan_edge_l1,
            "reg_weight":   reg_weight,
            "n_epochs":     300,
            "patience":     20,
        }
        if target_type == "continuous":
            train_kwargs["huber_delta"] = huber_delta

        return model, train_kwargs
    return factory


# ═══════════════════════════════════════════════════════════════════════════
# SINGLE EXPERIMENT
# ═══════════════════════════════════════════════════════════════════════════

def run_single_experiment(split_name, dataset, target_type, device,
                          seed, seed_results_dir):
    model_name = f"sparse_kan_{dataset}"

    print(f"\n{'─'*60}")
    print(f"  seed={seed} / {dataset} / {split_name} / {target_type}")
    print(f"{'─'*60}")

    data         = load_split(split_name, dataset, SPLITS_DIR)
    feature_cols = data["feature_cols"]
    taxonomy_df  = taxonomy_dfs[dataset]

    n_pos      = data["y_train"].sum()
    n_neg      = len(data["y_train"]) - n_pos
    pos_weight = torch.tensor([n_neg / max(n_pos, 1)], dtype=torch.float32)

    huber_delta = get_huber_delta(split_name) if target_type == "continuous" else None
    metric_name = "AUC" if target_type == "binary" else "R2"

    # ── Fixed probe batch for live activation monitoring during the final
    # retrain (see make_activation_callback above). Pure diagnostic --
    # does not affect training in any way. ──
    rng = np.random.default_rng(seed)
    n_avail = data["X_train"].shape[0]
    probe_idx = rng.choice(n_avail, size=min(ACTIVATION_PROBE_N, n_avail), replace=False)
    activation_probe = torch.tensor(data["X_train"][probe_idx], dtype=torch.float32)

    def run_optuna_phase(phase_name, factory, n_trials):
        study_name = (f"{model_name}_{target_type}_{split_name}"
                      f"_{phase_name}_seed{seed}")
        study_path = seed_results_dir / "optuna" / f"{study_name}.db"
        study_path.parent.mkdir(parents=True, exist_ok=True)

        study = optuna.create_study(
            study_name=study_name,
            storage=f"sqlite:///{study_path}",
            direction="maximize",
            load_if_exists=True,
            sampler=optuna.samplers.TPESampler(seed=seed),
        )

        def objective(trial):
            model, train_kwargs = factory(trial)
            batch_size = trial.params["batch_size"]

            loaders = get_dataloaders(
                split_name, dataset, SPLITS_DIR,
                target_type=target_type,
                batch_size=batch_size,
            )

            result = train_model(
                model=model,
                train_loader=loaders["train"],
                val_loader=loaders["val"],
                device=device,
                target_type=target_type,
                pos_weight=pos_weight if target_type == "binary" else None,
                verbose=False,       # epoch_callback never fires during search
                **train_kwargs,
            )

            return result["best_val_metric"]

        existing  = sum(1 for t in study.trials
                        if t.state == optuna.trial.TrialState.COMPLETE)
        remaining = max(0, n_trials - existing)

        if remaining > 0:
            print(f"  {phase_name}: {remaining} trials ({existing} already complete)")
            study.optimize(objective, n_trials=remaining, show_progress_bar=True)
        else:
            print(f"  {phase_name}: {existing} trials already complete — skipping")

        print(f"  {phase_name} best {metric_name}: {study.best_value:.4f}  "
              f"params: {study.best_params}")
        return study

    optuna.logging.set_verbosity(optuna.logging.WARNING)

    factory_no_l1 = make_model_factory_no_l1(feature_cols, taxonomy_df, huber_delta, target_type)
    study_no_l1 = run_optuna_phase("no_L1", factory_no_l1, N_TRIALS_NO_L1)

    factory_l1 = make_model_factory_with_l1(feature_cols, taxonomy_df, huber_delta, target_type)
    study_l1 = run_optuna_phase("with_L1", factory_l1, N_TRIALS)

    no_l1_wins = study_no_l1.best_value >= study_l1.best_value

    if no_l1_wins:
        best_params = study_no_l1.best_params
        best_val    = study_no_l1.best_value
        use_l1      = False
        print(f"\n  → No-L1 wins (seed={seed}, {metric_name}={best_val:.4f})")
    else:
        best_params = study_l1.best_params
        best_val    = study_l1.best_value
        use_l1      = True
        print(f"\n  → With-L1 wins (seed={seed}, {metric_name}={best_val:.4f})")

    # ── FINAL TRAINING with best params ──
    batch_size = best_params.get("batch_size", 128)
    loaders    = get_dataloaders(
        split_name, dataset, SPLITS_DIR,
        target_type=target_type,
        batch_size=batch_size,
    )

    model = SparseKAN.from_taxonomy(
        taxonomy_df, feature_cols,
        grid_size=GRID_SIZE, spline_order=SPLINE_ORDER, grid_range=GRID_RANGE,
    )
    activation_callback = make_activation_callback(activation_probe, device)

    final_train_kwargs = {
        "lr":              best_params["lr"],
        "weight_decay":    best_params["weight_decay"],
        "pos_weight":      pos_weight if target_type == "binary" else None,
        "n_epochs":        300,
        "patience":        20,
        "verbose":         True,       # activation_callback WILL fire, per log_every
        "log_every":       20,
        "epoch_callback":  activation_callback,
    }
    if use_l1:
        final_train_kwargs["reg_fn"]     = sparse_kan_edge_l1
        final_train_kwargs["reg_weight"] = best_params["reg_weight"]
    if target_type == "continuous":
        final_train_kwargs["huber_delta"] = huber_delta

    result = train_model(
        model=model,
        train_loader=loaders["train"],
        val_loader=loaders["val"],
        device=device,
        target_type=target_type,
        **final_train_kwargs,
    )

    # ── Post-training masking verification (belt-and-braces) ──
    assert model.verify_masking(), (
        "Masked weights non-zero after final training completed -- "
        "structural sparsity has been broken somewhere. Investigate "
        "before trusting this checkpoint."
    )

    all_metrics = {}
    for part in ["train", "val", "test"]:
        metrics = evaluate_model(
            model, loaders[part], device, target_type,
            y_true_binary=data[f"y_{part}"] if target_type == "continuous" else None,
        )

        if target_type == "binary":
            cal = compute_calibration(metrics["y_true"], metrics["y_prob"])
            metrics["ece"] = cal["ece"]

        save_predictions(
            model_name=model_name,
            split_name=split_name,
            target_type=target_type,
            part=part,
            dates=data[f"dates_{part}"],
            returns=data[f"returns_{part}"],
            metrics=metrics,
            hyperparameters=(
                {**best_params, "used_l1": use_l1, "seed": seed,
                 "huber_delta": huber_delta}
                if part == "test" else None
            ),
            results_dir=seed_results_dir,
            y_true_binary=data[f"y_{part}"] if target_type == "continuous" else None,
        )

        all_metrics[part] = metrics

    ckpt_dir = seed_results_dir / "checkpoints"
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    save_checkpoint(
        model=model,
        train_result=result,
        hyperparameters={**best_params, "used_l1": use_l1, "seed": seed,
                         "huber_delta": huber_delta},
        model_config={
            "type":                "SparseKAN_Masked_BN",
            "dataset":             dataset,
            "target_type":         target_type,
            "n_features":          data["n_features"],
            "n_subthemes":         model.n_subthemes,
            "n_themes":            model.n_themes,
            "grid_size":           GRID_SIZE,
            "spline_order":        SPLINE_ORDER,
            "grid_range":          GRID_RANGE,
            "active_edges":        model.count_active_edges(),
            "total_edges":         model.count_total_edges(),
            "active_parameters":   model.count_active_parameters(),
        },
        path=ckpt_dir / f"{model_name}_{target_type}_{split_name}.pt",
    )

    l1_str = (f"reg_weight={best_params['reg_weight']:.2e}"
              if use_l1 else "no L1")
    if target_type == "binary":
        print(f"\n  Results (seed={seed}, {l1_str}):")
        print(f"    Train AUC: {all_metrics['train']['auc']:.4f}")
        print(f"    Val AUC:   {all_metrics['val']['auc']:.4f}")
        print(f"    Test AUC:  {all_metrics['test']['auc']:.4f}")
        print(f"    Gap:       "
              f"{all_metrics['train']['auc'] - all_metrics['test']['auc']:+.4f}")
    else:
        print(f"\n  Results (seed={seed}, {l1_str}, huber_delta={huber_delta:.4f}):")
        print(f"    Train R2:    {all_metrics['train']['r2']:.4f}  "
              f"(MSE={all_metrics['train']['mse']:.4f})")
        print(f"    Val R2:      {all_metrics['val']['r2']:.4f}  "
              f"(MSE={all_metrics['val']['mse']:.4f})")
        print(f"    Test R2:     {all_metrics['test']['r2']:.4f}  "
              f"(MSE={all_metrics['test']['mse']:.4f})")
        print(f"    Derived AUC: {all_metrics['test']['derived_auc']:.4f}")
        print(f"    Pred std:    {all_metrics['test']['pred_std']:.4f}  "
              f"(collapse check deferred to Collapse_Audit.ipynb)")

    return {
        "best_params": best_params,
        "used_l1":     use_l1,
        "metrics":     all_metrics,
        "best_epoch":  result["best_epoch"],
        "total_time":  result["total_time"],
    }


# %% [markdown]
# ## Architecture Summary (from live model counts)

# %%
print("=" * 70)
print("  SPARSE KAN ARCHITECTURE SUMMARY")
print("=" * 70 + "\n")
for ds, info in architecture_info.items():
    print(f"  {ds}:")
    print(f"    Layer widths:      [{info['n_features']}, {info['n_subthemes']}, "
          f"{info['n_themes']}, 1]")
    print(f"    Active edges:      {info['active_edges']:,} / {info['total_edges']:,} "
          f"({info['active_edges']/info['total_edges']*100:.2f}%)")
    print(f"    Active parameters: {info['active_parameters']:,}")
    print()


# %% [markdown]
# ## Run All Experiments (3 Seeds × 16 Configurations)

# %%
configs    = len(DATASETS) * len(TARGET_TYPES) * len(ALL_SPLITS)
total_runs = len(SEEDS) * configs

print("=" * 70)
print(f"  SPARSE KAN: 3 Seeds × 4 Splits × 2 Datasets × 2 Targets = {total_runs} runs")
print(f"  Seeds: {SEEDS}")
print(f"  Grid: G={GRID_SIZE}, K={SPLINE_ORDER}, range={GRID_RANGE} "
      f"(FIXED, no grid adaptation, all layers)")
print(f"  BatchNorm1d between every KAN layer (see sparse_kan.py)")
print(f"  Loss: binary=BCEWithLogitsLoss, "
      f"continuous=HuberLoss(delta=per-split, see huber_delta.json)")
print(f"  Early stop: binary=AUC, continuous=R2  (both maximize)")
print(f"  Optuna Phase 1: {N_TRIALS_NO_L1} trials (no L1), seeded TPE sampler")
print(f"  Optuna Phase 2: {N_TRIALS} trials (with L1 [1e-7, 1e-3]), seeded TPE sampler")
print(f"  Collapse handling: detect-and-flag post-hoc (Collapse_Audit.ipynb), no retry")
print(f"  Results saving to: {RESULTS_DIR}/seed_*/")
print("=" * 70)

all_results       = []
best_params_store = {}
completed         = 0
failed            = 0
total_start       = time.time()

for seed in SEEDS:
    set_seed(seed)
    seed_results_dir = RESULTS_DIR / f"seed_{seed}"

    print(f"\n\n{'═'*70}")
    print(f"  SEED {seed} — saving to {seed_results_dir}")
    print(f"{'═'*70}")

    for dataset in DATASETS:
        for target_type in TARGET_TYPES:
            for split_name in ALL_SPLITS:
                try:
                    exp = run_single_experiment(
                        split_name, dataset, target_type, device,
                        seed=seed,
                        seed_results_dir=seed_results_dir,
                    )

                    all_results.append({
                        "seed":    seed,
                        "dataset": dataset,
                        "split":   split_name,
                        "target":  target_type,
                        "used_l1": exp["used_l1"],
                        **{f"test_{k}": v for k, v in exp["metrics"]["test"].items()
                           if not isinstance(v, np.ndarray)},
                        "best_epoch": exp["best_epoch"],
                        "time_s":     exp["total_time"],
                    })

                    key = (seed, dataset, target_type, split_name)
                    best_params_store[key] = {
                        **exp["best_params"], "used_l1": exp["used_l1"],
                    }
                    completed += 1

                    elapsed       = time.time() - total_start
                    rate          = elapsed / completed
                    remaining_est = rate * (total_runs - completed)
                    print(f"\n  ✓ Completed {completed}/{total_runs}  "
                          f"({elapsed/60:.0f}min elapsed, "
                          f"~{remaining_est/60:.0f}min remaining)")

                except Exception as e:
                    failed += 1
                    print(f"\n  ✗ FAILED ({failed}): seed={seed} "
                          f"{dataset}/{split_name}/{target_type}: {e}")
                    import traceback
                    traceback.print_exc()
                    continue

total_time = time.time() - total_start
print(f"\n\n{'='*70}")
print(f"  FINISHED: {completed}/{total_runs} completed, {failed} failed")
print(f"  Total time: {total_time/60:.1f} minutes ({total_time/3600:.1f} hours)")
print(f"{'='*70}")


# %% [markdown]
# ## Cross-Seed Summary
#
# The collapse audit (Collapse_Audit.ipynb) must be run AFTER this notebook
# finishes. Once it exists, re-run this cell to exclude flagged runs.

# %%
if all_results:
    results_df = pd.DataFrame(all_results)

    audit_path = RESULTS_DIR.parent / "collapse_audit.csv"
    if audit_path.exists():
        audit_df = pd.read_csv(audit_path)
        sub_audit = audit_df[audit_df.model == "sparse_kan"][
            ["dataset", "target_type", "split", "seed", "collapsed"]
        ].rename(columns={"target_type": "target"})
        results_df = results_df.merge(
            sub_audit, on=["dataset", "target", "split", "seed"], how="left"
        )
        results_df["collapsed"] = results_df["collapsed"].fillna(False)
        n_dropped = results_df["collapsed"].sum()
        if n_dropped:
            print(f"  Excluding {n_dropped} collapsed run(s) from aggregation "
                  f"(per collapse_audit.csv)")
        results_df = results_df[~results_df["collapsed"]].drop(columns="collapsed")
    else:
        print(f"  collapse_audit.csv not found at {audit_path} -- "
              f"run Collapse_Audit.ipynb after this notebook completes, "
              f"then re-run this cell.")

    raw_path = RESULTS_DIR / "all_seeds_raw.csv"
    results_df.to_csv(raw_path, index=False)
    print(f"  Raw results saved to {raw_path}")

    binary_df = results_df[results_df["target"] == "binary"]

    print("\n" + "=" * 70)
    print("  SPARSE KAN — Binary Test AUC (mean ± std across seeds)")
    print("=" * 70 + "\n")

    if "test_auc" in binary_df.columns and len(binary_df) > 0:
        agg = binary_df.groupby(["dataset", "split"])["test_auc"].agg(
            ["mean", "std"]
        ).reset_index()
        agg["display"] = agg.apply(
            lambda r: f"{r['mean']:.4f} ± {r['std']:.4f}", axis=1
        )
        pivot = agg.pivot(index="dataset", columns="split", values="display")
        grand = binary_df.groupby("dataset")["test_auc"].agg(["mean", "std"])
        pivot["Mean ± Std"] = grand.apply(
            lambda r: f"{r['mean']:.4f} ± {r['std']:.4f}", axis=1
        )
        print(pivot.to_string())

    cont_df = results_df[results_df["target"] == "continuous"]

    print("\n" + "=" * 70)
    print("  SPARSE KAN — Continuous Test R2 (mean ± std across seeds)")
    print("=" * 70 + "\n")

    if "test_r2" in cont_df.columns and len(cont_df) > 0:
        agg = cont_df.groupby(["dataset", "split"])["test_r2"].agg(
            ["mean", "std"]
        ).reset_index()
        agg["display"] = agg.apply(
            lambda r: f"{r['mean']:.4f} ± {r['std']:.4f}", axis=1
        )
        pivot = agg.pivot(index="dataset", columns="split", values="display")
        grand = cont_df.groupby("dataset")["test_r2"].agg(["mean", "std"])
        pivot["Mean ± Std"] = grand.apply(
            lambda r: f"{r['mean']:.4f} ± {r['std']:.4f}", axis=1
        )
        print(pivot.to_string())

    print("\n" + "=" * 70)
    print("  SPARSE KAN — Continuous Derived AUC (mean ± std across seeds)")
    print("=" * 70 + "\n")

    if "test_derived_auc" in cont_df.columns and len(cont_df) > 0:
        agg = cont_df.groupby(["dataset", "split"])["test_derived_auc"].agg(
            ["mean", "std"]
        ).reset_index()
        agg["display"] = agg.apply(
            lambda r: f"{r['mean']:.4f} ± {r['std']:.4f}", axis=1
        )
        pivot = agg.pivot(index="dataset", columns="split", values="display")
        grand = cont_df.groupby("dataset")["test_derived_auc"].agg(["mean", "std"])
        pivot["Mean ± Std"] = grand.apply(
            lambda r: f"{r['mean']:.4f} ± {r['std']:.4f}", axis=1
        )
        print(pivot.to_string())

    print("\n" + "=" * 70)
    print("  SPARSE KAN — Continuous Test MSE (mean ± std across seeds)")
    print("=" * 70 + "\n")

    if "test_mse" in cont_df.columns and len(cont_df) > 0:
        agg = cont_df.groupby(["dataset", "split"])["test_mse"].agg(
            ["mean", "std"]
        ).reset_index()
        agg["display"] = agg.apply(
            lambda r: f"{r['mean']:.4f} ± {r['std']:.4f}", axis=1
        )
        pivot = agg.pivot(index="dataset", columns="split", values="display")
        grand = cont_df.groupby("dataset")["test_mse"].agg(["mean", "std"])
        pivot["Mean ± Std"] = grand.apply(
            lambda r: f"{r['mean']:.4f} ± {r['std']:.4f}", axis=1
        )
        print(pivot.to_string())
        print(f"\n  NOTE: MSE is in minret_5d_pct units (percentage points squared).")

    print("\n" + "=" * 70)
    print("  CONTINUOUS: Prediction std (informational)")
    print("=" * 70 + "\n")

    if "test_pred_std" in cont_df.columns:
        for _, row in cont_df.iterrows():
            print(f"  seed={row['seed']}  {row['dataset']:<20} "
                  f"{row['split']:<10}  "
                  f"pred_std={row.get('test_pred_std', 0):.4f}")

    print("\n" + "=" * 70)
    print("  L1 USAGE: Did the Sparse KAN need L1?")
    print("=" * 70 + "\n")

    if "used_l1" in results_df.columns:
        n_l1  = results_df["used_l1"].sum()
        n_tot = len(results_df)
        print(f"  L1 selected: {n_l1}/{n_tot} runs across all seeds")
        for tt in TARGET_TYPES:
            subset = results_df[results_df["target"] == tt]
            n      = subset["used_l1"].sum()
            print(f"    {tt}: {n}/{len(subset)} runs used L1")
        print()
        if n_l1 == 0:
            print("  → Structural sparsity alone sufficient — L1 never needed")
        elif n_l1 == n_tot:
            print("  → L1 always helps — structural sparsity alone insufficient")
        else:
            print("  → Mixed — L1 helps for some configurations but not others")

    print("\n" + "=" * 70)
    print("  SEED STABILITY: Best hyperparameters across seeds")
    print("=" * 70 + "\n")

    print(f"  {'Seed':>5} {'Dataset':<20} {'Target':<12} {'Split':<10} "
          f"{'lr':>10} {'wd':>10} {'bs':>5} {'reg_w':>10} {'L1?':>4}")
    print("  " + "-" * 90)
    for (seed, ds, tt, split), params in sorted(best_params_store.items()):
        rw   = params.get("reg_weight", 0)
        used = params.get("used_l1", False)
        print(f"  {seed:>5} {ds:<20} {tt:<12} {split:<10} "
              f"{params['lr']:>10.6f} {params['weight_decay']:>10.6f} "
              f"{params['batch_size']:>5} {rw:>10.2e} "
              f"{'yes' if used else 'no':>4}")

    print("\n" + "=" * 70)
    print("  SEED VARIANCE SUMMARY")
    print("=" * 70 + "\n")

    for tt in TARGET_TYPES:
        metric_col   = "test_auc"         if tt == "binary" else "test_derived_auc"
        metric_label = "AUC"              if tt == "binary" else "Derived AUC"
        subset = results_df[results_df["target"] == tt]

        if metric_col not in subset.columns or len(subset) == 0:
            continue

        print(f"  {tt.upper()} ({metric_label}):")
        per_config = subset.groupby(["dataset", "split"])[metric_col].agg(
            ["mean", "std"]
        )
        max_std  = per_config["std"].max()
        mean_std = per_config["std"].mean()
        print(f"    Mean seed std across configs: {mean_std:.4f}")
        print(f"    Max seed std across configs:  {max_std:.4f}")

        if max_std < 0.01:
            print(f"    → Very stable: seed choice barely matters")
        elif max_std < 0.03:
            print(f"    → Moderately stable: some sensitivity to initialisation")
        else:
            print(f"    → High variance: results depend substantially on seed")
        print()

    print("\n" + "=" * 70)
    print("  TIMING")
    print("=" * 70 + "\n")

    for _, row in results_df.iterrows():
        print(f"  seed={row['seed']}  {row['dataset']:<20} "
              f"{row['split']:<10} {row['target']:<12} "
              f"best_epoch={row['best_epoch']:>3}  {row['time_s']:>6.1f}s")

    summary_rows = []
    for tt in TARGET_TYPES:
        subset = results_df[results_df["target"] == tt]
        if len(subset) == 0:
            continue

        metric_cols = [c for c in subset.columns
                       if c.startswith("test_") and
                       subset[c].dtype in [np.float64, np.float32, float]]

        agg = subset.groupby(["dataset", "split"])[metric_cols].agg(
            ["mean", "std"]
        ).reset_index()

        agg.columns = [
            f"{c[0]}_{c[1]}" if c[1] else c[0]
            for c in agg.columns
        ]

        agg["target"] = tt
        summary_rows.append(agg)

    if summary_rows:
        summary_df   = pd.concat(summary_rows, ignore_index=True)
        summary_path = RESULTS_DIR / "cross_seed_summary.csv"
        summary_df.to_csv(summary_path, index=False)
        print(f"\n  Cross-seed summary saved to {summary_path}")

else:
    print("\n  No results to display — all experiments failed.")


# %% [markdown]
# ## File Inventory

# %%
print("\n" + "=" * 70)
print("  SAVED FILES (on Google Drive)")
print("=" * 70)

for seed in SEEDS:
    seed_dir = RESULTS_DIR / f"seed_{seed}"
    print(f"\n  ── seed_{seed}/ ──")
    for subdir in ["predictions", "metrics", "checkpoints", "optuna"]:
        d = seed_dir / subdir
        if d.exists():
            files = list(d.glob("sparse_kan_*"))
            print(f"    {subdir}/: {len(files)} files")
        else:
            print(f"    {subdir}/: (not yet created)")

for fname in ["all_seeds_raw.csv", "cross_seed_summary.csv"]:
    fpath = RESULTS_DIR / fname
    if fpath.exists():
        print(f"\n  {fname}: ✓")
    else:
        print(f"\n  {fname}: (not yet created)")


# %% [markdown]
# ## Backtests (Seed-Averaged Signal)
#
# NOTE: evaluation.py's fixed continuous threshold grids still assume a
# roughly [-5, +2] z-score range from the old target — grid-edge warnings
# here reflect that KNOWN, DEFERRED issue, not a new bug.

# %%
def load_averaged_predictions(model_name, split_name, target_type, part,
                               seeds, results_dir):
    signals = []
    returns = None

    for seed in seeds:
        seed_dir = results_dir / f"seed_{seed}"
        loaded = load_predictions(
            model_name=model_name,
            split_name=split_name,
            target_type=target_type,
            part=part,
            results_dir=seed_dir,
        )
        preds = loaded["predictions"]

        if returns is None:
            returns = preds["daily_return"].values

        if target_type == "binary":
            signals.append(preds["y_prob"].values)
        else:
            signals.append(preds["y_pred"].values)

    avg_signal = np.mean(np.stack(signals, axis=0), axis=0)
    return returns, avg_signal


def _make_json_safe(obj):
    if isinstance(obj, dict): return {k: _make_json_safe(v) for k, v in obj.items()}
    if isinstance(obj, pd.DataFrame): return obj.reset_index().to_dict(orient="records")
    if isinstance(obj, (np.floating, np.integer)): return obj.item()
    if isinstance(obj, np.ndarray): return obj.tolist()
    return obj


# %%
print("\n" + "=" * 70)
print("  SPARSE KAN — BACKTESTS (seed-averaged signal)")
print("  Signal = mean prediction across seeds 42, 123, 456")
print("=" * 70)

backtest_dir = RESULTS_DIR / "backtests"
backtest_dir.mkdir(parents=True, exist_ok=True)

backtest_rows    = []
backtest_records = {}

for dataset in DATASETS:
    for split_name in ALL_SPLITS:
        model_name = f"sparse_kan_{dataset}"

        val_ret,  val_sig  = load_averaged_predictions(
            model_name, split_name, "binary", "val",  SEEDS, RESULTS_DIR)
        test_ret, test_sig = load_averaged_predictions(
            model_name, split_name, "binary", "test", SEEDS, RESULTS_DIR)

        bt_binary = run_full_backtest(
            val_returns=val_ret,   val_signal=val_sig,
            test_returns=test_ret, test_signal=test_sig,
            go_cash_when="above",
            model_name=f"{model_name} (binary)",
            split_name=split_name,
        )

        val_ret,  val_sig  = load_averaged_predictions(
            model_name, split_name, "continuous", "val",  SEEDS, RESULTS_DIR)
        test_ret, test_sig = load_averaged_predictions(
            model_name, split_name, "continuous", "test", SEEDS, RESULTS_DIR)

        bt_continuous = run_full_backtest(
            val_returns=val_ret,   val_signal=val_sig,
            test_returns=test_ret, test_signal=test_sig,
            go_cash_when="below",
            model_name=f"{model_name} (continuous)",
            split_name=split_name,
        )

        key = f"{dataset}/{split_name}"
        backtest_records[key] = {"binary": bt_binary, "continuous": bt_continuous}

        for target_type, bt in [("binary", bt_binary), ("continuous", bt_continuous)]:
            for strategy_name, strategy_key in [("simple", "simple"), ("risk_scaled", "risk_scaled")]:
                bt_result = bt[strategy_key]
                backtest_rows.append({
                    "dataset": dataset, "split": split_name,
                    "target_type": target_type, "strategy": strategy_name,
                    "sharpe": bt_result["sharpe"], "sortino": bt_result["sortino"],
                    "annual_return": bt_result["annual_return"],
                    "max_drawdown": bt_result["max_drawdown"],
                    "cumulative_return": bt_result["cumulative_return"],
                    "avg_exposure": bt_result["avg_exposure"],
                    "annual_turnover": bt_result["annual_turnover"],
                    "buy_hold_sharpe": bt_result["buy_hold_sharpe"],
                    "buy_hold_sortino": bt_result["buy_hold_sortino"],
                    "buy_hold_cumulative": bt_result["buy_hold_cumulative"],
                })

backtest_summary_df = pd.DataFrame(backtest_rows)
backtest_summary_path = backtest_dir / "backtest_summary.csv"
backtest_summary_df.to_csv(backtest_summary_path, index=False)
print(f"\n  Backtest summary saved to {backtest_summary_path}")

backtest_json_path = backtest_dir / "backtest_full_results.json"
with open(backtest_json_path, "w") as f:
    json.dump(_make_json_safe(backtest_records), f, indent=2, default=str)
print(f"  Full backtest results saved to {backtest_json_path}")


# %% [markdown]
# ## Disconnect Runtime

# %%
print("All experiments complete. Disconnecting runtime...")
from google.colab import runtime
runtime.unassign()

Mounted at /content/drive
PyTorch: 2.11.0+cu128
CUDA: True
GPU: Tesla T4
SANITY CHECK: verify_masking() NaN blind spot -- confirming the FIX
  Injecting a real NaN at a masked position and calling the live
  verify_masking(). Before the fix this returned True (silently
  'clear'). After the fix it must return False.
  ERROR: Layer 0 base_weight has 1 non-finite and/or non-zero masked entries (max finite abs=0.00e+00)

  verify_masking() on a model with an injected masked NaN: False
  ✓ verify_masking() correctly detects the injected NaN
  ✓ Forward pass with BatchNorm in train mode works (batch size 16)

SANITY CHECK PASSED
Loading taxonomies...
  agg_full_moments: 1699 features, 331 subthemes, 13 themes
  agg_means: 574 features, 128 subthemes, 13 themes
Device: Tesla T4 (CUDA)
  ✓ All masked parameters are exactly zero and finite
  agg_full_moments: 2,043 active edges / 566,685 total  (38,817 active params) — ✓ GPU forward pass OK, masking verified pre-training
  ✓ All masked paramet

  0%|          | 0/30 [00:00<?, ?it/s]

  no_L1 best AUC: 0.7697  params: {'lr': 0.00010994335574766199, 'weight_decay': 0.00757947995334801, 'batch_size': 64}
  with_L1: 40 trials (0 already complete)


  0%|          | 0/40 [00:00<?, ?it/s]

  with_L1 best AUC: 0.8078  params: {'lr': 0.0004066563313514797, 'weight_decay': 2.4586032763280077e-06, 'batch_size': 64, 'reg_weight': 9.565499215943809e-06}

  → With-L1 wins (seed=42, AUC=0.8078)
  Epoch    1 | Train loss 1.1015 | Val loss 0.7560  AUC 0.6431 | LR 4.1e-04
    [activations @ epoch 1]  L1 pre-BN max|x|=   1.43 std= 0.044  post-BN(pre-clamp) max|x|= 3.78 std=0.112  clamp_rate=0.000%  |  L2 pre-BN max|x|=   0.27 std= 0.025  post-BN(pre-clamp) max|x|= 0.76 std=0.069  clamp_rate=0.000%
  Epoch   20 | Train loss 0.7249 | Val loss 0.4422  AUC 0.6883 | LR 2.0e-04
    [activations @ epoch 20]  L1 pre-BN max|x|=   2.37 std= 0.154  post-BN(pre-clamp) max|x|= 5.20 std=0.359  clamp_rate=0.000%  |  L2 pre-BN max|x|=   1.73 std= 0.328  post-BN(pre-clamp) max|x|= 3.57 std=0.677  clamp_rate=0.000%
  Epoch   40 | Train loss 0.5790 | Val loss 0.4202  AUC 0.6857 | LR 1.0e-04
    [activations @ epoch 40]  L1 pre-BN max|x|=   2.06 std= 0.150  post-BN(pre-clamp) max|x|= 5.46 std=0.372  cl

...

  with_L1 best R2: 0.3009  params: {'lr': 0.0002074699474505998, 'weight_decay': 0.00010866420630921273, 'batch_size': 64, 'reg_weight': 0.0003062985587276315}

  → With-L1 wins (seed=456, R2=0.3009)
  Epoch    1 | Train Huber 1.1254 | Val Huber 1.5659  MSE 4.6041  R² -0.6305 | LR 2.1e-04
    [activations @ epoch 1]  L1 pre-BN max|x|=   0.92 std= 0.043  post-BN(pre-clamp) max|x|= 2.51 std=0.113  clamp_rate=0.000%  |  L2 pre-BN max|x|=   0.24 std= 0.013  post-BN(pre-clamp) max|x|= 0.74 std=0.039  clamp_rate=0.000%
  Epoch   20 | Train Huber 0.3718 | Val Huber 0.8018  MSE 2.2151  R² 0.2155 | LR 2.1e-04
    [activations @ epoch 20]  L1 pre-BN max|x|=   3.32 std= 0.176  post-BN(pre-clamp) max|x|= 6.32 std=0.365  clamp_rate=0.004%  |  L2 pre-BN max|x|=   2.42 std= 0.234  post-BN(pre-clamp) max|x|= 4.44 std=0.500  clamp_rate=0.000%
  Epoch   40 | Train Huber 0.3313 | Val Huber 0.8043  MSE 2.1562  R² 0.2364 | LR 2.1e-04
    [activations @ epoch 40]  L1 pre-BN max|x|=   3.21 std= 0.166  post-B

  0%|          | 0/30 [00:00<?, ?it/s]

  no_L1 best R2: 0.2235  params: {'lr': 0.00015164597276518462, 'weight_decay': 7.875185124692694e-06, 'batch_size': 64}
  with_L1: 40 trials (0 already complete)


  0%|          | 0/40 [00:00<?, ?it/s]

  with_L1 best R2: 0.2701  params: {'lr': 0.003696579594060844, 'weight_decay': 4.2769488927038806e-06, 'batch_size': 128, 'reg_weight': 0.0009454494392293611}

  → With-L1 wins (seed=456, R2=0.2701)
  Epoch    1 | Train Huber 1.0844 | Val Huber 1.1473  MSE 2.3656  R² -1.1051 | LR 3.7e-03
    [activations @ epoch 1]  L1 pre-BN max|x|=   3.27 std= 0.129  post-BN(pre-clamp) max|x|= 5.47 std=0.245  clamp_rate=0.002%  |  L2 pre-BN max|x|=   1.19 std= 0.094  post-BN(pre-clamp) max|x|= 2.84 std=0.226  clamp_rate=0.000%
  Epoch   20 | Train Huber 0.2844 | Val Huber 0.5333  MSE 1.0628  R² 0.0542 | LR 3.7e-03
    [activations @ epoch 20]  L1 pre-BN max|x|=   2.00 std= 0.098  post-BN(pre-clamp) max|x|= 4.67 std=0.234  clamp_rate=0.000%  |  L2 pre-BN max|x|=   1.74 std= 0.243  post-BN(pre-clamp) max|x|= 3.90 std=0.464  clamp_rate=0.000%
  Early stop at epoch 34. Best val R²: 0.1784 at epoch 14
  Training complete in 10.1s
  ✓ All masked parameters are exactly zero and finite

  Results (seed=456,